# World Development Indicators data set

The goal of this notebook is to investigate influential outliers in the WDI data set.

In [1]:
import shap

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import seaborn as sns
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor

In [2]:
from main_iom import *

In [3]:
sns.set_theme(style='white')

We read in the data and select a few variables. Our goal is to predict the GDP in 2022 of a country given captial, savings, population growth and consumption.

In [4]:
df = pd.read_csv("WDICSV.csv")
df = df[["Country Name", "Country Code", "Indicator Name", "Indicator Code", "2022"]]

In [5]:
df_final = df[df["Indicator Name"].isin(["GDP per capita (current US$)", 
                                         "Gross capital formation (current US$)", 
                                         "Population growth (annual %)", 
                                         "Gross savings (current US$)", 
                                         "Final consumption expenditure (current US$)"])] \
                                    .pivot(columns='Indicator Name', 
                                           index='Country Name', 
                                           values='2022') \
                                    .dropna()

In [6]:
df_final.columns.name= None
df_final.index.name = None

We set up the data to be used in the model

In [7]:
X = df_final.drop("GDP per capita (current US$)", axis=1)
y = df_final['GDP per capita (current US$)']

We set up a randomized grid search fro hyperparameter tuning.

In [8]:
grid = {'learning_rate': [0.03, 0.1],
        'depth': [4, 6, 10],
        'l2_leaf_reg': [1, 3, 5, 7, 9]}

In [9]:
mod = CatBoostRegressor(random_seed=123)

result = mod.randomized_search(grid, X=X, y=y, n_iter=10)

0:	learn: 31383.8874624	test: 27471.5803819	best: 27471.5803819 (0)	total: 133ms	remaining: 2m 12s
1:	learn: 30903.0483510	test: 27244.4746903	best: 27244.4746903 (1)	total: 134ms	remaining: 1m 6s
2:	learn: 30412.9742965	test: 27042.1942270	best: 27042.1942270 (2)	total: 135ms	remaining: 44.8s
3:	learn: 29947.3514475	test: 26838.4170843	best: 26838.4170843 (3)	total: 136ms	remaining: 33.7s
4:	learn: 29511.9030865	test: 26635.6146715	best: 26635.6146715 (4)	total: 136ms	remaining: 27.1s
5:	learn: 29141.9429118	test: 26494.3993695	best: 26494.3993695 (5)	total: 137ms	remaining: 22.7s
6:	learn: 28684.9280290	test: 26336.8161747	best: 26336.8161747 (6)	total: 137ms	remaining: 19.5s
7:	learn: 28306.4197211	test: 26168.0753282	best: 26168.0753282 (7)	total: 138ms	remaining: 17.1s
8:	learn: 27934.7978331	test: 26026.7287331	best: 26026.7287331 (8)	total: 139ms	remaining: 15.3s
9:	learn: 27572.2164484	test: 25874.3361701	best: 25874.3361701 (9)	total: 140ms	remaining: 13.8s
10:	learn: 27206.01

We verify the fit of the model using cross validation.

In [10]:
pd.DataFrame(result['cv_results']).loc[19]

iterations            19.000000
test-RMSE-mean     26904.300910
test-RMSE-std       2723.628305
train-RMSE-mean    26167.676456
train-RMSE-std      1026.770901
Name: 19, dtype: float64

The final parameters chosen.

In [11]:
result['params']

{'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 7}

We apply the `catboost` regressor and calulate the SHAP values and residuals.

In [12]:
explainer = shap.TreeExplainer(mod, X)
shap_values = explainer.shap_values(X)

In [13]:
resid = y.to_numpy() - mod.predict(X)

We test if the normality assumption is valid for the SHAP values and residuals.

In [14]:
print(f"SHAP values: {multivariate_normality(shap_values).pval:.4f}")
print(f"Residuals: {stats.shapiro(resid.reshape(-1)).pvalue:.4f}")

SHAP values: 0.0000
Residuals: 0.0000


They reject the test for normality.

In [15]:
iom_cat = InfluentialOutlierMetric(shap_values, resid,
                                    6, 2, 6, 
                                    1, 1, 2, 
                                    lambdas=np.concatenate([[0], np.exp(np.linspace(-5, 5, 50))]),
                                    lambdas_resid=np.concatenate([[0], np.exp(np.linspace(-5, 5, 50))]),
                                    epoch=500, epoch_resid=500)

In [16]:
# Tune for lambda
iom_cat.find_best_lambda(alpha=0.05)
iom_cat.find_best_lambda_resid(alpha=0.05)

# Compute thresholds
thresholds = iom_cat.find_threshold(alpha=[0.05, 0.01])

# Compute IOM
IOM_cat = iom_cat.IOM()

Finding λ for SHAP values
Henze–Zirkler p-value = 0.0000
λ=0.0000


Normalizing flow:   9%|▉         | 44/500 [00:06<01:03,  7.22it/s]

Normalizing flow: 100%|██████████| 500/500 [01:02<00:00,  7.98it/s]


p=0.3450
---
λ=0.0067


Normalizing flow: 100%|██████████| 500/500 [00:56<00:00,  8.87it/s]


...

Normalizing flow: 100%|██████████| 500/500 [00:07<00:00, 68.79it/s]


p=0.0556
---
λ=0.1439


Normalizing flow: 100%|██████████| 500/500 [00:07<00:00, 67.65it/s]

p=0.0444
---
Selected λ=0.1173, p=0.0556
Done!
Threshold at 0.05: 16.9168
Threshold at 0.01: 35.8830



c:\Users\jonc\OneDrive - Bank of Canada\jonc\PhD\main_iom.py:312: IntegrationWarning: The algorithm does not converge.  Roundoff error is detected
  in the extrapolation table.  It is assumed that the requested tolerance
  cannot be achieved, and that the returned result (if full_output = 1) is 
  the best which can be obtained.
  val, _ = quad(chi_prod, i, np.inf, args=(m1, m2))


Put it all together.

In [17]:
X_df = pd.DataFrame(X, index=X.index, columns=X.columns)
X_df[["Final consumption expenditure (current US$B)", 
            "Gross capital formation (current US$B)",
            "Gross savings (current US$B)"]] = X_df[["Final consumption expenditure (current US$)", 
                                                          "Gross capital formation (current US$)",	
                                                          "Gross savings (current US$)"]]*1e-9

X_df = X_df.drop(columns=["Final consumption expenditure (current US$)", 
                                "Gross capital formation (current US$)",	
                                "Gross savings (current US$)"])

In [18]:
iom_cat.IOM_.index = X.index
X_df["Outlier"] = np.where(iom_cat.IOM_ > iom_cat.i_star[1], "Influential outlier", "Inlier")

In [19]:
Final = pd.concat([X_df,
                   pd.Series(y, name="GDP per capita (current US$)"),
                   pd.Series(iom_cat.chi_z.reshape(-1), name='SHAP leverage (transformed)', index=X.index),
                   pd.Series(iom_cat.chi_resid.reshape(-1), name='Residual squared (transformed)', index=X.index),
                   iom_cat.IOM_],
            axis=1)

In [20]:
round(Final[Final["Outlier"]=="Influential outlier"], 2).drop(columns=["Outlier"]).sort_values("IOM", ascending=False).round(2)

,Population growth (annual %),Final consumption expenditure (current US$B),Gross capital formation (current US$B),Gross savings (current US$B),GDP per capita (current US$),SHAP leverage (transformed),Residual squared (transformed),IOM
Luxembourg,2.02,39.36,14.49,14.91,125006.02,16.36,15.23,249.23
Iceland,2.51,21.80,7.01,6.42,75314.09,6.19,8.98,55.58


In [21]:
iom_cat.summary()

               Parameter      Value
             Samples (n) 150.000000
            Features (p)   4.000000
       Flow depth (SHAP)   6.000000
     Hidden units (SHAP)   6.000000
           Layers (SHAP)   2.000000
  Flow depth (Residuals)   1.000000
Hidden units (Residuals)   2.000000
      Layers (Residuals)   1.000000
          Final λ (SHAP)   0.028116
            Shap p-value   0.109956
     Final λ (Residuals)   0.117319
        Residual p-value   0.055648
      Threshold (α=0.05)  16.916796
      Threshold (α=0.01)  35.883027


,Parameter,Value
0,Samples (n),150.000000
1,Features (p),4.000000
2,Flow depth (SHAP),6.000000
3,Hidden units (SHAP),6.000000
4,Layers (SHAP),2.000000
5,Flow depth (Residuals),1.000000
6,Hidden units (Residuals),2.000000
7,Layers (Residuals),1.000000
8,Final λ (SHAP),0.028116
9,Shap p-value,0.109956
